In [15]:
from pathlib import Path

In [16]:
results_json = sorted(Path("GPU4EDU outputs").rglob("*.json"))

In [17]:
import json
from statistics import mean

TEMPORAL_KEYS = ("0.1", "0.25", "0.5")
METRIC_KEYS = ("top1", "top5", "macro_precision", "macro_recall", "macro_f1")

def load_json(path: Path):
    with open(path, "r") as f:
        return json.load(f)

def _condition_payload(data: dict, condition: str):
    payload = data["temporal_conditions"][condition]
    return {
        "top1": payload["top1"],
        "top5": payload["top5"],
        "macro_precision": payload["macro_precision"],
        "macro_recall": payload["macro_recall"],
        "macro_f1": payload["macro_f1"],
        "per_class": payload["per_class"],
    }

def extract_condition_metrics(path: Path, condition: str):
    data = load_json(path)
    return _condition_payload(data, condition)

def init_bucket():
    return {
        "top1": [],
        "top5": [],
        "macro_precision": [],
        "macro_recall": [],
        "macro_f1": [],
        "per_class": [],
    }

def add_run(bucket: dict, metrics: dict):
    for key in METRIC_KEYS:
        bucket[key].append(metrics[key])
    bucket["per_class"].append(metrics["per_class"])

def avg_numeric(values):
    return round(mean(values), 4) if values else None

In [18]:
results = {
    "resnet": {
        "direct": {k: init_bucket() for k in TEMPORAL_KEYS},
        "full_to_limited": {k: init_bucket() for k in TEMPORAL_KEYS},
    },
    "timesformer": {
        "direct": {k: init_bucket() for k in TEMPORAL_KEYS},
        "full_to_limited": {k: init_bucket() for k in TEMPORAL_KEYS},
    },
}

for result in results_json:
    model, split, _ = result.stem.split("_")

    if split == "full":
        data = load_json(result)
        for condition in TEMPORAL_KEYS:
            add_run(results[model]["full_to_limited"][condition], _condition_payload(data, condition))
    else:
        add_run(results[model]["direct"][split], extract_condition_metrics(result, split))

In [19]:
for model in ("resnet", "timesformer"):
    for experiment in ("direct", "full_to_limited"):
        for condition in TEMPORAL_KEYS:
            run_count = len(results[model][experiment][condition]["top1"])
            print(f"{model}_{experiment}_{condition} run_count: {run_count}")

resnet_direct_0.1 run_count: 3
resnet_direct_0.25 run_count: 3
resnet_direct_0.5 run_count: 3
resnet_full_to_limited_0.1 run_count: 3
resnet_full_to_limited_0.25 run_count: 3
resnet_full_to_limited_0.5 run_count: 3
timesformer_direct_0.1 run_count: 3
timesformer_direct_0.25 run_count: 3
timesformer_direct_0.5 run_count: 3
timesformer_full_to_limited_0.1 run_count: 3
timesformer_full_to_limited_0.25 run_count: 3
timesformer_full_to_limited_0.5 run_count: 3


In [20]:
import pandas as pd

def summarize_macro(bucket: dict):
    return {key: avg_numeric(bucket[key]) for key in METRIC_KEYS}

def summarize_per_class(per_class_runs: list[dict]):
    class_accumulator = {}

    for run in per_class_runs:
        for class_name, metrics in run.items():
            class_accumulator.setdefault(
                class_name,
                {"precision": [], "recall": [], "f1": [], "support": []},
            )
            class_accumulator[class_name]["precision"].append(metrics["precision"])
            class_accumulator[class_name]["recall"].append(metrics["recall"])
            class_accumulator[class_name]["f1"].append(metrics["f1"])
            class_accumulator[class_name]["support"].append(metrics["support"])

    summarized = {}
    for class_name, metric_lists in class_accumulator.items():
        summarized[class_name] = {
            "precision": round(mean(metric_lists["precision"]), 4),
            "recall": round(mean(metric_lists["recall"]), 4),
            "f1": round(mean(metric_lists["f1"]), 4),
            "support": int(round(mean(metric_lists["support"]))),
        }
    return summarized

def per_class_table(per_class_metrics: dict, sort_by: str = "f1", ascending: bool = False):
    rows = [
        {"class": class_name, **metrics}
        for class_name, metrics in per_class_metrics.items()
    ]
    return pd.DataFrame(rows).sort_values(by=sort_by, ascending=ascending).reset_index(drop=True)

def top_classes(per_class_metrics: dict, by: str = "f1", n: int = 10, reverse: bool = True):
    rows = [
        (class_name, metrics["precision"], metrics["recall"], metrics["f1"], metrics["support"])
        for class_name, metrics in per_class_metrics.items()
    ]
    index = {"precision": 1, "recall": 2, "f1": 3}[by]
    rows.sort(key=lambda item: item[index], reverse=reverse)
    return rows[:n]

EXPERIMENTS = ("direct", "full_to_limited")
MODELS = ("resnet", "timesformer")

macro_summary = {model: {exp: {} for exp in EXPERIMENTS} for model in MODELS}
per_class_summary = {model: {exp: {} for exp in EXPERIMENTS} for model in MODELS}
per_class_tables = {model: {exp: {} for exp in EXPERIMENTS} for model in MODELS}

for model in MODELS:
    print(f"\n=== {model.upper()} ===")
    for experiment in EXPERIMENTS:
        print(f"\n  [{experiment}]")
        for condition in TEMPORAL_KEYS:
            bucket = results[model][experiment][condition]
            macro_summary[model][experiment][condition] = summarize_macro(bucket)
            per_class_summary[model][experiment][condition] = summarize_per_class(bucket["per_class"])
            per_class_tables[model][experiment][condition] = per_class_table(
                per_class_summary[model][experiment][condition], sort_by="f1", ascending=False
            )

            print(f"    {condition}: {macro_summary[model][experiment][condition]}")

            print(f"    Top 10 classes by F1 ({model}, {experiment}, {condition}):")
            for class_name, precision, recall, f1, support in top_classes(
                per_class_summary[model][experiment][condition], by="f1", n=10
            ):
                print(
                    f"      {class_name:<22} P={precision:.3f} R={recall:.3f} F1={f1:.3f} support={support}"
                )

print("\nPer-class DataFrames are available in per_class_tables[model][experiment][condition].")
print("Example: per_class_tables['resnet']['direct']['0.1'].head(20)")


=== RESNET ===

  [direct]
    0.1: {'top1': 65.8931, 'top5': 84.8566, 'macro_precision': 0.6755, 'macro_recall': 0.6573, 'macro_f1': 0.6479}
    Top 10 classes by F1 (resnet, direct, 0.1):
      Billiards              P=0.983 R=0.968 F1=0.976 support=43
      HorseRace              P=0.943 R=0.961 F1=0.949 support=35
      BasketballDunk         P=0.898 R=0.983 F1=0.938 support=38
      PlayingGuitar          P=0.917 R=0.926 F1=0.918 support=46
      PlayingSitar           P=0.935 R=0.908 F1=0.918 support=44
      Diving                 P=0.910 R=0.896 F1=0.903 support=42
      IceDancing             P=0.858 R=0.954 F1=0.902 support=43
      Punch                  P=0.906 R=0.894 F1=0.898 support=44
      SoccerPenalty          P=0.943 R=0.860 F1=0.898 support=39
      SumoWrestling          P=0.936 R=0.869 F1=0.897 support=33
    0.25: {'top1': 68.9447, 'top5': 88.0295, 'macro_precision': 0.7132, 'macro_recall': 0.6886, 'macro_f1': 0.6811}
    Top 10 classes by F1 (resnet, direct, 0

In [21]:
export_dir = Path("analysis_exports")
export_dir.mkdir(parents=True, exist_ok=True)

macro_rows = []
for model in MODELS:
    for experiment in EXPERIMENTS:
        for condition in TEMPORAL_KEYS:
            row = {
                "model": model,
                "experiment": experiment,
                "condition": condition,
                **macro_summary[model][experiment][condition],
            }
            macro_rows.append(row)

macro_df = pd.DataFrame(macro_rows).sort_values(
    by=["model", "experiment", "condition"]
).reset_index(drop=True)
macro_path = export_dir / "macro_summary.csv"
macro_df.to_csv(macro_path, index=False)

per_class_rows = []
for model in MODELS:
    for experiment in EXPERIMENTS:
        for condition in TEMPORAL_KEYS:
            class_metrics = per_class_summary[model][experiment][condition]
            for class_name, metrics in class_metrics.items():
                per_class_rows.append(
                    {
                        "model": model,
                        "experiment": experiment,
                        "condition": condition,
                        "class": class_name,
                        "precision": metrics["precision"],
                        "recall": metrics["recall"],
                        "f1": metrics["f1"],
                        "support": metrics["support"],
                    }
                )

per_class_df = pd.DataFrame(per_class_rows).sort_values(
    by=["model", "experiment", "condition", "class"]
).reset_index(drop=True)
per_class_path = export_dir / "per_class_metrics.csv"
per_class_df.to_csv(per_class_path, index=False)

print(f"Saved: {macro_path}")
print(f"Saved: {per_class_path}")

Saved: analysis_exports\macro_summary.csv
Saved: analysis_exports\per_class_metrics.csv


## Model Comparison Visuals (Charts 1-5)
This section generates the requested figures from the aggregated results.
Chart 6 (confusion delta) is skipped because confusion-level prediction pairs are not available in the saved JSON outputs.

In [22]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

sns.set_theme(style="white", context="paper")

INK = "#1A1917"
INK_SOFT = "#3D3B37"
INK_MUTED = "#7A776E"
INK_FAINT = "#B8B5AD"
PAPER = "#F7F5F0"
PAPER_WARM = "#EEEBE3"

ACCENT = "#1B3A5C"
ACCENT_MID = "#2C5F8A"
ACCENT_LIGHT = "#D6E6F3"
ACCENT_SUBTLE = "#EBF3FA"

DATA_1 = "#1B3A5C"
DATA_2 = "#C0522B"
DATA_3 = "#2E7D5E"
DATA_4 = "#7A5C99"
DATA_5 = "#B08A2E"

FONT_SERIF = "Instrument Serif"
FONT_SANS = "DM Sans"
FONT_MONO = "DM Mono"

mpl.rcParams.update({
    "font.family": FONT_SANS,
    "font.size": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.labelsize": 9,
    "axes.labelcolor": INK_SOFT,
    "axes.titlesize": 11,
    "axes.titlecolor": INK,
    "axes.edgecolor": INK_MUTED,
    "axes.linewidth": 1.0,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "xtick.color": INK_MUTED,
    "ytick.color": INK_MUTED,
    "grid.color": INK_FAINT,
    "grid.linewidth": 0.5,
    "grid.linestyle": (0, (3, 3)),
    "legend.fontsize": 9,
    "legend.frameon": False,
    "figure.dpi": 300,
    "figure.facecolor": PAPER,
    "axes.facecolor": PAPER_WARM,
    "savefig.format": "pdf",
    "savefig.bbox": "tight",
    "savefig.facecolor": PAPER,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "lines.solid_capstyle": "round",
    "lines.dash_capstyle": "round",
})

FIG_DPI = 300
FIGSIZE_SINGLE = (10.5, 5.6)
FIGSIZE_DOUBLE = (13.5, 5.6)

MODEL_STYLE = {
    "resnet": {"color": DATA_1, "marker": "o"},
    "timesformer": {"color": DATA_2, "marker": "s"},
}

EXPERIMENT_STYLE = {
    "direct": {"linestyle": "-", "linewidth": 2.0, "alpha": 1.0, "marker_size": 6, "edge_width": 1.8},
    "full_to_limited": {
        "linestyle": (0, (6, 4)),
        "linewidth": 1.5,
        "alpha": 0.6,
        "marker_size": 5,
        "edge_width": 1.4,
    },
}

def add_bottom_legend(fig, handles, labels, ncol=2):
    fig.legend(
        handles,
        labels,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.01),
        ncol=ncol,
        frameon=False,
        prop={"family": FONT_SANS, "size": 9},
    )

def apply_axis_style(ax, title=None, xlabel=None, ylabel=None):
    if title:
        ax.set_title(title, fontfamily=FONT_SERIF, fontsize=11, color=INK, pad=10)
    if xlabel:
        ax.set_xlabel(xlabel, fontfamily=FONT_SANS, fontsize=9, color=INK_SOFT, labelpad=8)
    if ylabel:
        ax.set_ylabel(ylabel, fontfamily=FONT_SANS, fontsize=9, color=INK_SOFT, labelpad=8)
    ax.tick_params(axis="both", colors=INK_MUTED, labelsize=8)
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontfamily(FONT_MONO)
    for spine in ax.spines.values():
        spine.set_color(INK_MUTED)
        spine.set_linewidth(1.0)

def ensure_pdf_path(path: Path) -> Path:
    return path.with_suffix(".pdf")

def finalize_plot(fig, out_path, with_legend=False, use_tight_layout=True):
    out_path = ensure_pdf_path(out_path)
    if use_tight_layout:
        if with_legend:
            fig.tight_layout(rect=[0, 0.08, 1, 0.98])
        else:
            fig.tight_layout(rect=[0, 0.02, 1, 0.98])
    else:
        if with_legend:
            fig.subplots_adjust(bottom=0.10, top=0.95)
        else:
            fig.subplots_adjust(bottom=0.06, top=0.95)

    fig.savefig(out_path, dpi=FIG_DPI)
    plt.close(fig)
    print(f"Saved: {out_path}")

fraction_to_pct = {"0.1": 10, "0.25": 25, "0.5": 50}
model_label = {"resnet": "ResNet-18", "timesformer": "TimeSformer"}
experiment_label = {"direct": "Matched Train", "full_to_limited": "Full Train"}

viz_dir = Path("analysis_exports") / "figures"
viz_dir.mkdir(parents=True, exist_ok=True)

metric_rows = []
for model in MODELS:
    for experiment in EXPERIMENTS:
        for condition in TEMPORAL_KEYS:
            row = {
                "model": model,
                "model_label": model_label[model],
                "experiment": experiment,
                "experiment_label": experiment_label[experiment],
                "condition": condition,
                "obs_pct": fraction_to_pct[condition],
                **macro_summary[model][experiment][condition],
            }
            metric_rows.append(row)

metrics_df = pd.DataFrame(metric_rows).sort_values(
    by=["model", "experiment", "obs_pct"]
).reset_index(drop=True)

print("Prepared metrics_df for plotting:")
display(metrics_df.head(10))
print(f"Figures will be saved to: {viz_dir}")

Prepared metrics_df for plotting:


,model,model_label,experiment,experiment_label,condition,obs_pct,top1,top5,macro_precision,macro_recall,macro_f1
0,resnet,ResNet-18,direct,Matched Train,0.1,10,65.8931,84.8566,0.6755,0.6573,0.6479
1,resnet,ResNet-18,direct,Matched Train,0.25,25,68.9447,88.0295,0.7132,0.6886,0.6811
2,resnet,ResNet-18,direct,Matched Train,0.5,50,70.5378,88.7624,0.7312,0.7033,0.6995
3,resnet,ResNet-18,full_to_limited,Full Train,0.1,10,61.0018,82.5255,0.6619,0.6081,0.5994
4,resnet,ResNet-18,full_to_limited,Full Train,0.25,25,64.7269,84.7959,0.6838,0.6465,0.6373
5,resnet,ResNet-18,full_to_limited,Full Train,0.5,50,68.1303,86.9518,0.7151,0.6816,0.6726
6,timesformer,TimeSformer,direct,Matched Train,0.1,10,70.3467,88.5262,0.7493,0.6998,0.6979
7,timesformer,TimeSformer,direct,Matched Train,0.25,25,74.2700,90.4261,0.7867,0.7404,0.7390
8,timesformer,TimeSformer,direct,Matched Train,0.5,50,77.0368,91.7055,0.7957,0.7703,0.7669
9,timesformer,TimeSformer,full_to_limited,Full Train,0.1,10,68.4266,87.4702,0.7472,0.6789,0.6839


Figures will be saved to: analysis_exports\figures


In [23]:
# Chart 1: Matched condition Top-1 line chart
matched_df = metrics_df[metrics_df["experiment"] == "direct"].copy()

fig, ax = plt.subplots(figsize=FIGSIZE_SINGLE)
for model in ["resnet", "timesformer"]:
    part = matched_df[matched_df["model"] == model].sort_values("obs_pct")
    style = MODEL_STYLE[model]
    exp_style = EXPERIMENT_STYLE["direct"]

    ax.fill_between(
        part["obs_pct"],
        part["top1"],
        color=style["color"],
        alpha=0.08,
        zorder=1,
    )
    ax.plot(
        part["obs_pct"],
        part["top1"],
        color=style["color"],
        marker=style["marker"],
        linestyle=exp_style["linestyle"],
        linewidth=exp_style["linewidth"],
        markersize=exp_style["marker_size"],
        markerfacecolor="white",
        markeredgecolor=style["color"],
        markeredgewidth=exp_style["edge_width"],
        label=model_label[model],
        zorder=3,
    )

ax.set_xticks([10, 25, 50], ["10%", "25%", "50%"] )
ax.margins(x=0.05)
ax.grid(alpha=0.2)
ax.set_axisbelow(True)
apply_axis_style(
    ax,
    title="Matched Condition Top-1 Accuracy",
    xlabel="Observed Video Fraction",
    ylabel="Top-1 Accuracy (%)",
)

handles, labels = ax.get_legend_handles_labels()
add_bottom_legend(fig, handles, labels, ncol=2)

chart1_path = viz_dir / "chart1_matched_top1_line.pdf"
finalize_plot(fig, chart1_path, with_legend=True)

findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Sans' not found.
findfont: Font family 'DM Sans' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Sans' not found.
findfont: Font family 'Instrument Serif' not found.
findfont: Font family 'Instrument Serif' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Sans' not 

Saved: analysis_exports\figures\chart1_matched_top1_line.pdf


In [24]:
# Chart 2: Distribution shift cost grouped bars (matched vs full train)
fig, axes = plt.subplots(1, 2, figsize=FIGSIZE_DOUBLE, sharey=True)
obs_labels = [10, 25, 50]
x = np.arange(len(obs_labels))
width = 0.36

matched_color = DATA_1
full_color = DATA_2

for idx, model in enumerate(["resnet", "timesformer"]):
    ax = axes[idx]
    model_df = metrics_df[metrics_df["model"] == model]

    matched_vals = (
        model_df[model_df["experiment"] == "direct"]
        .sort_values("obs_pct")["top1"]
        .to_numpy()
    )
    full_vals = (
        model_df[model_df["experiment"] == "full_to_limited"]
        .sort_values("obs_pct")["top1"]
        .to_numpy()
    )

    ax.bar(
        x - width / 2,
        matched_vals,
        width=width,
        label=experiment_label["direct"],
        color=matched_color,
        alpha=0.9,
    )
    ax.bar(
        x + width / 2,
        full_vals,
        width=width,
        label=experiment_label["full_to_limited"],
        color=full_color,
        alpha=0.9,
    )

    ax.set_xticks(x, [f"{v}%" for v in obs_labels])
    ax.set_ylim(0, 100)
    ax.grid(axis="y", alpha=0.2)
    ax.set_axisbelow(True)
    ax.margins(x=0.1)
    apply_axis_style(
        ax,
        title=model_label[model],
        xlabel="Observed Video Fraction",
    )

apply_axis_style(axes[0], ylabel="Top-1 Accuracy (%)")
handles, labels = axes[0].get_legend_handles_labels()
add_bottom_legend(fig, handles, labels, ncol=2)
fig.suptitle(
    "Distribution Shift Cost (Condition Mismatch)",
    fontfamily=FONT_SERIF,
    fontsize=11,
    color=INK,
)

chart2_path = viz_dir / "chart2_distribution_shift_grouped_bar.pdf"
finalize_plot(fig, chart2_path, with_legend=True)

findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Sans' not found.
findfont: Font family 'DM Sans' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Sans' not found.
findfont: Font family 'Instrument Serif' not found.
findfont: Font family 'Instrument Serif' not found.
findfont: Font family 'Instrument Serif' not found.
findfont: Font family 'Instrument Serif' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Sans' not found.
findfont: Font fam

Saved: analysis_exports\figures\chart2_distribution_shift_grouped_bar.pdf


In [25]:
# Chart 3: Gap chart (TimeSformer - ResNet)
gap_rows = []
obs_labels = [10, 25, 50]
for experiment in EXPERIMENTS:
    for obs in obs_labels:
        ts = metrics_df[
            (metrics_df["model"] == "timesformer")
            & (metrics_df["experiment"] == experiment)
            & (metrics_df["obs_pct"] == obs)
        ]["top1"].iloc[0]
        rn = metrics_df[
            (metrics_df["model"] == "resnet")
            & (metrics_df["experiment"] == experiment)
            & (metrics_df["obs_pct"] == obs)
        ]["top1"].iloc[0]
        gap_rows.append(
            {
                "experiment": experiment,
                "experiment_label": experiment_label[experiment],
                "obs_pct": obs,
                "gap_pp": ts - rn,
            }
        )

gap_df = pd.DataFrame(gap_rows)

fig, ax = plt.subplots(figsize=FIGSIZE_SINGLE)
x = np.arange(len(obs_labels))
width = 0.36

exp_colors = {"direct": DATA_1, "full_to_limited": DATA_2}
for i, exp in enumerate(["direct", "full_to_limited"]):
    vals = gap_df[gap_df["experiment"] == exp].sort_values("obs_pct")["gap_pp"].to_numpy()
    offset = (-width / 2) if i == 0 else (width / 2)
    ax.bar(
        x + offset,
        vals,
        width=width,
        label=experiment_label[exp],
        color=exp_colors[exp],
        alpha=0.9,
    )

ax.axhline(0, color=INK_MUTED, linewidth=1.0)
ax.set_xticks(x, [f"{v}%" for v in obs_labels])
ax.set_xlim(-0.6, len(obs_labels) - 0.4)
ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)
apply_axis_style(
    ax,
    title="TimeSformer Advantage Over ResNet",
    xlabel="Observed Video Fraction",
    ylabel="Top-1 Gap (pp)",
)

handles, labels = ax.get_legend_handles_labels()
add_bottom_legend(fig, handles, labels, ncol=2)

chart3_path = viz_dir / "chart3_gap_pp_grouped_bar.pdf"
finalize_plot(fig, chart3_path, with_legend=True)

findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Sans' not found.
findfont: Font family 'DM Sans' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Sans' not found.
findfont: Font family 'Instrument Serif' not found.
findfont: Font family 'Instrument Serif' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Sans' not 

Saved: analysis_exports\figures\chart3_gap_pp_grouped_bar.pdf


In [26]:
# Chart 4: Top-5 comparison with Top-1 context
fig, axes = plt.subplots(1, 2, figsize=FIGSIZE_DOUBLE, sharey=True)

for ax, exp in zip(axes, ["direct", "full_to_limited"]):
    part = metrics_df[metrics_df["experiment"] == exp]
    for model in ["resnet", "timesformer"]:
        model_part = part[part["model"] == model].sort_values("obs_pct")
        style = MODEL_STYLE[model]
        exp_style = EXPERIMENT_STYLE["direct"] if exp == "direct" else EXPERIMENT_STYLE["full_to_limited"]

        ax.plot(
            model_part["obs_pct"],
            model_part["top1"],
            marker=style["marker"],
            linestyle=exp_style["linestyle"],
            color=style["color"],
            linewidth=exp_style["linewidth"],
            markersize=exp_style["marker_size"],
            markerfacecolor="white",
            markeredgecolor=style["color"],
            markeredgewidth=exp_style["edge_width"],
            label=f"{model_label[model]} Top-1",
        )
        ax.plot(
            model_part["obs_pct"],
            model_part["top5"],
            marker=style["marker"],
            linestyle=(0, (6, 4)),
            color=style["color"],
            linewidth=1.5,
            markersize=exp_style["marker_size"] - 0.5,
            markerfacecolor="white",
            markeredgecolor=style["color"],
            markeredgewidth=1.3,
            alpha=0.6,
            label=f"{model_label[model]} Top-5",
        )

    ax.set_xticks([10, 25, 50], ["10%", "25%", "50%"] )
    ax.grid(alpha=0.2)
    ax.set_axisbelow(True)
    apply_axis_style(ax, title=experiment_label[exp], xlabel="Observed Video Fraction")

apply_axis_style(axes[0], ylabel="Accuracy (%)")
handles, labels = axes[0].get_legend_handles_labels()
add_bottom_legend(fig, handles, labels, ncol=2)
fig.suptitle(
    "Top-1 vs Top-5 Accuracy by Observation Level",
    fontfamily=FONT_SERIF,
    fontsize=11,
    color=INK,
)

chart4_path = viz_dir / "chart4_top1_top5_panel.pdf"
finalize_plot(fig, chart4_path, with_legend=True)

findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Sans' not found.
findfont: Font family 'DM Sans' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Sans' not found.
findfont: Font family 'Instrument Serif' not found.
findfont: Font family 'Instrument Serif' not found.
findfont: Font family 'Instrument Serif' not found.
findfont: Font family 'Instrument Serif' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font fam

Saved: analysis_exports\figures\chart4_top1_top5_panel.pdf


In [27]:
# Chart 5: Per-class F1 (top-20 and bottom-20 at 10%) + unlock heatmaps
res10 = per_class_summary["resnet"]["direct"]["0.1"]
ts10 = per_class_summary["timesformer"]["direct"]["0.1"]

res10_df = pd.DataFrame(
    [{"class": c, "resnet_f1": m["f1"]} for c, m in res10.items()]
).set_index("class")
ts10_df = pd.DataFrame(
    [{"class": c, "timesformer_f1": m["f1"]} for c, m in ts10.items()]
).set_index("class")

combined10 = res10_df.join(ts10_df, how="inner")
combined10["gap_pp"] = (combined10["timesformer_f1"] - combined10["resnet_f1"]) * 100

res_low20 = set(combined10.nsmallest(20, "resnet_f1").index)
res_high20 = set(combined10.nlargest(20, "resnet_f1").index)
ts_low20 = set(combined10.nsmallest(20, "timesformer_f1").index)
ts_high20 = set(combined10.nlargest(20, "timesformer_f1").index)
selected_classes = sorted(
    res_low20 | res_high20 | ts_low20 | ts_high20,
    key=lambda c: combined10.loc[c, "timesformer_f1"],
)

selected_df = combined10.loc[selected_classes].reset_index()

fig, ax = plt.subplots(figsize=(12.5, max(9, 0.28 * len(selected_df))))
y = np.arange(len(selected_df))
bar_h = 0.38
ax.barh(
    y + bar_h / 2,
    selected_df["resnet_f1"],
    height=bar_h,
    label="ResNet-18",
    color=MODEL_STYLE["resnet"]["color"],
    alpha=0.9,
    zorder=3,
)
ax.barh(
    y - bar_h / 2,
    selected_df["timesformer_f1"],
    height=bar_h,
    label="TimeSformer",
    color=MODEL_STYLE["timesformer"]["color"],
    alpha=0.9,
    zorder=3,
)
ax.set_yticks(y)
ax.set_yticklabels(selected_df["class"], fontsize=8)
ax.tick_params(axis="y", pad=2)
ax.set_xlim(0, 1.0)
ax.grid(axis="x", alpha=0.2)
ax.set_axisbelow(True)
apply_axis_style(
    ax,
    title="Chart 5A: Per-class F1 at 10% (Top/Bottom sets from both models)",
    xlabel="F1 at 10% observed fraction (matched)",
)
for label in ax.get_yticklabels():
    label.set_fontfamily(FONT_MONO)

handles, labels = ax.get_legend_handles_labels()
add_bottom_legend(fig, handles, labels, ncol=2)

chart5a_path = viz_dir / "chart5a_per_class_f1_sorted_bar.pdf"
finalize_plot(fig, chart5a_path, with_legend=True)

def build_heatmap_df(model_name: str, classes: list[str]):
    rows = []
    for class_name in classes:
        row = {"class": class_name}
        for cond in ["0.1", "0.25", "0.5"]:
            row[f"{fraction_to_pct[cond]}%"] = per_class_summary[model_name]["direct"][cond][class_name]["f1"]
        rows.append(row)
    heat = pd.DataFrame(rows).set_index("class")
    return heat

heat_res = build_heatmap_df("resnet", selected_classes)
heat_ts = build_heatmap_df("timesformer", selected_classes)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(14.0, max(9, 0.28 * len(selected_classes))),
    sharey=True,
    gridspec_kw={"wspace": 0.04},
)
heatmap_kwargs = {
    "cmap": sns.light_palette(DATA_1, as_cmap=True),
    "vmin": 0,
    "vmax": 1,
    "linewidths": 0.2,
    "linecolor": PAPER,
}
sns.heatmap(heat_res, ax=axes[0], cbar=False, **heatmap_kwargs)
axes[0].set_title("ResNet-18", fontfamily=FONT_SERIF, fontsize=11, color=INK)
axes[0].set_xlabel("Observed fraction", fontfamily=FONT_SANS, fontsize=9, color=INK_SOFT)
axes[0].set_ylabel("Class", fontfamily=FONT_SANS, fontsize=9, color=INK_SOFT)
axes[0].tick_params(axis="y", labelsize=8)
axes[0].tick_params(axis="x", labelsize=8)
for label in axes[0].get_xticklabels() + axes[0].get_yticklabels():
    label.set_fontfamily(FONT_MONO)

sns.heatmap(
    heat_ts,
    ax=axes[1],
    cbar_kws={"label": "F1", "shrink": 0.85, "pad": 0.02},
    **heatmap_kwargs,
)
axes[1].set_title("TimeSformer", fontfamily=FONT_SERIF, fontsize=11, color=INK)
axes[1].set_xlabel("Observed fraction", fontfamily=FONT_SANS, fontsize=9, color=INK_SOFT)
axes[1].set_ylabel("")
axes[1].tick_params(axis="x", labelsize=8)
for label in axes[1].get_xticklabels():
    label.set_fontfamily(FONT_MONO)

fig.suptitle(
    "Chart 5B: Class Unlock Pattern Across 10/25/50% (Matched)",
    y=0.98,
    fontfamily=FONT_SERIF,
    fontsize=11,
    color=INK,
)
fig.subplots_adjust(top=0.92, wspace=0.04, bottom=0.06)

chart5b_path = viz_dir / "chart5b_per_class_f1_heatmap.pdf"
chart5b_path = ensure_pdf_path(chart5b_path)
fig.savefig(chart5b_path, dpi=FIG_DPI, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {chart5b_path}")

findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Sans' not found.
findfont: Font family 'DM Sans' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: F

Saved: analysis_exports\figures\chart5a_per_class_f1_sorted_bar.pdf


findfont: Font family 'DM Sans' not found.
findfont: Font family 'DM Sans' not found.
findfont: Font family 'DM Sans' not found.
findfont: Font family 'DM Sans' not found.
findfont: Font family 'DM Sans' not found.
findfont: Font family 'DM Sans' not found.
findfont: Font family 'DM Sans' not found.
findfont: Font family 'DM Sans' not found.
findfont: Font family 'DM Sans' not found.
findfont: Font family 'DM Sans' not found.
findfont: Font family 'DM Sans' not found.
findfont: Font family 'DM Sans' not found.
findfont: Font family 'DM Sans' not found.
findfont: Font family 'DM Sans' not found.
findfont: Font family 'DM Sans' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Sans' not found.
findfont: Font family 'DM Sans' not found.
findfont: Font family 'DM Mono' not found.
findfont: Font family 'DM Mono' not found.
findfont: F

Saved: analysis_exports\figures\chart5b_per_class_f1_heatmap.pdf
